<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Misc/IPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import time
import io
import requests
import pandas as pd
import yfinance as yf

years = [2024, 2025, 2026]

# Mimic a standard browser handshake to prevent 403 forbidden responses
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5"
}

print("Initiating StockAnalysis IPO data retrieval pipeline...")

all_yearly_dfs = [] # List to store DataFrames for each year

for year in years:
    # Target URL pattern for historical yearly listings
    url = f"https://stockanalysis.com/ipos/{year}/"
    print(f"Fetching: {url}")

    try:
        response = requests.get(url, headers=HEADERS, timeout=15)

        if response.status_code == 200:
            # Wrap the raw HTML text in a StringIO block to satisfy newer pandas requirements
            tables = pd.read_html(io.StringIO(response.text))

            if tables:
                # The primary historical data grid is the first table on the page
                df_year = tables[0]
                all_yearly_dfs.append(df_year) # Add the current year's DataFrame to the list

                # Transform the data frame into a structured JSON payload
                raw_payload = {
                    "year": year,
                    "source": "stockanalysis.com",
                    "count": len(df_year),
                    "ipos": df_year.to_dict(orient="records")
                }

                output_filename = f"/content/ipo_state_{year}.json"
                with open(output_filename, "w", encoding="utf-8") as f:
                    json.dump(raw_payload, f, ensure_ascii=False, indent=4)

                print(f"     ✅ Successfully saved {len(df_year)} listings to: {output_filename}")
            else:
                print(f"     ❌ Table element not found in HTML structure for year {year}.")
        else:
            print(f"     ❌ Failed connection to {year}. HTTP Status: {response.status_code}")

    except Exception as e:
        print(f"     ❌ Scraping exception encountered for year {year}: {str(e)}")

    # 2-second decay delay to be a polite scraper and prevent rate limits
    time.sleep(2.0)

# Concatenate all yearly DataFrames into a single DataFrame
if all_yearly_dfs:
    consolidated_df = pd.concat(all_yearly_dfs, ignore_index=True)
    print("\n--- Consolidated DataFrame Head Sample ---")
    display(consolidated_df.head())
    print("\n--- Consolidated DataFrame Tail Sample ---")
    display(consolidated_df.tail())
else:
    print("\nNo data was retrieved to consolidate.")

print("\nRetrieval step complete. Proceed to Cell 2 to process.")

Initiating StockAnalysis IPO data retrieval pipeline...
Fetching: https://stockanalysis.com/ipos/2024/
     ✅ Successfully saved 225 listings to: /content/ipo_state_2024.json
Fetching: https://stockanalysis.com/ipos/2025/
     ✅ Successfully saved 347 listings to: /content/ipo_state_2025.json
Fetching: https://stockanalysis.com/ipos/2026/
     ✅ Successfully saved 199 listings to: /content/ipo_state_2026.json

--- Consolidated DataFrame Head Sample ---


,IPO Date,Symbol,Company Name,IPO Price,Current,Return
0,"Dec 31, 2024",ONEG,OneConstruction Group Limited,$4.00,$1.15,-71.25%
1,"Dec 27, 2024",BYAH,"Park Ha Biological Technology Co., Ltd.",-,$0.468,-
2,"Dec 23, 2024",HIT,"Health In Tech, Inc.",$4.00,$1.02,-73.50%
3,"Dec 23, 2024",TDAC,Translational Development Acquisition Corp.,$10.00,$10.71,7.10%
4,"Dec 20, 2024",RANG,Range Capital Acquisition Corp.,$10.00,$10.67,6.70%



--- Consolidated DataFrame Tail Sample ---


,IPO Date,Symbol,Company Name,IPO Price,Current,Return
766,"Jan 8, 2026",BBCQ,Bleichroeder Acquisition Corp. II,$10.00,$10.33,3.30%
767,"Jan 8, 2026",BUDA,"Buda Juice, Inc.",$7.50,$10.99,44.80%
768,"Jan 7, 2026",SORN,Soren Acquisition Corp.,$10.00,$9.94,-0.60%
769,"Jan 6, 2026",ARTC,Art Technology Acquisition Corp.,$10.00,$9.98,-0.20%
770,"Jan 6, 2026",BIII,Black Spade Acquisition III Co,$10.00,$9.94,-0.60%



Retrieval step complete. Proceed to Cell 2 to process.


### Loading and Merging 'IPO Deal Size' Data

Now, let's load the provided `IPO_Deal_Size.csv` file and merge its information (Deal Size) into our `consolidated_df`. We will assume the 'Symbol' column is present in both DataFrames for the merge operation.

In [2]:
# Load the IPO_Deal_Size.txt file as a pipe-separated file, handling quoted fields
ipo_deal_size_df = pd.read_csv('/content/IPO_Deal_Size.txt', sep='|', quotechar='"', skipinitialspace=True)

print("--- IPO Deal Size DataFrame Head ---")
display(ipo_deal_size_df.head())

--- IPO Deal Size DataFrame Head ---


,Symbol,Exchange,IPO Price,Shares Offered,Deal Size
0,RACD,NASDAQ,$10.00,"7,500,000",75.00M
1,SAMO,NYSE,$10.00,"20,000,000",200.00M
2,SKHY,NASDAQ,$149.00,"177,900,000",28.13B
3,CCCT,NASDAQ,$10.00,"20,000,000",200.00M
4,MRCO,NASDAQ,$10.00,"15,000,000",150.00M


In [3]:
# Merge the ipo_deal_size_df with consolidated_df on the 'Symbol' column
# We will use a left merge to keep all original IPOs from consolidated_df
consolidated_df_with_deal_size = pd.merge(
    consolidated_df,
    ipo_deal_size_df[['Symbol', 'Shares Offered', 'Deal Size']],
    on='Symbol',
    how='left'
)

print("--- Consolidated DataFrame with Deal Size (Head) ---")
display(consolidated_df_with_deal_size.head())

print("--- Consolidated DataFrame with Deal Size (Tail) ---")
display(consolidated_df_with_deal_size.tail())

print("\nMerge complete. The `consolidated_df_with_deal_size` DataFrame now includes 'Deal Size' and 'Shares Offered'.")

--- Consolidated DataFrame with Deal Size (Head) ---


,IPO Date,Symbol,Company Name,IPO Price,Current,Return,Shares Offered,Deal Size
0,"Dec 31, 2024",ONEG,OneConstruction Group Limited,$4.00,$1.15,-71.25%,"1,750,000",7.00M
1,"Dec 27, 2024",BYAH,"Park Ha Biological Technology Co., Ltd.",-,$0.468,-,-,-
2,"Dec 23, 2024",HIT,"Health In Tech, Inc.",$4.00,$1.02,-73.50%,"2,300,000",9.20M
3,"Dec 23, 2024",TDAC,Translational Development Acquisition Corp.,$10.00,$10.71,7.10%,"15,000,000",150.00M
4,"Dec 20, 2024",RANG,Range Capital Acquisition Corp.,$10.00,$10.67,6.70%,"10,000,000",100.00M


--- Consolidated DataFrame with Deal Size (Tail) ---


,IPO Date,Symbol,Company Name,IPO Price,Current,Return,Shares Offered,Deal Size
768,"Jan 8, 2026",BBCQ,Bleichroeder Acquisition Corp. II,$10.00,$10.33,3.30%,"25,000,000",250.00M
769,"Jan 8, 2026",BUDA,"Buda Juice, Inc.",$7.50,$10.99,44.80%,"2,666,667",20.00M
770,"Jan 7, 2026",SORN,Soren Acquisition Corp.,$10.00,$9.94,-0.60%,"22,000,000",220.00M
771,"Jan 6, 2026",ARTC,Art Technology Acquisition Corp.,$10.00,$9.98,-0.20%,"22,000,000",220.00M
772,"Jan 6, 2026",BIII,Black Spade Acquisition III Co,$10.00,$9.94,-0.60%,"15,000,000",150.00M



Merge complete. The `consolidated_df_with_deal_size` DataFrame now includes 'Deal Size' and 'Shares Offered'.


### Analyze and Calculate Returns for Large Deal Sizes

First, I will filter the `consolidated_df_with_deal_size` DataFrame to include only those IPOs where the 'Deal Size' contains 'B' (indicating billions). Then, for each of these filtered IPOs, I will use the `yfinance` library to fetch the stock's historical data for the IPO date. Finally, I will calculate the return based on the 'Open' and 'Close' prices for that specific date.

In [9]:
# Filter for rows where 'Deal Size' contains 'B'
filtered_df = consolidated_df_with_deal_size[consolidated_df_with_deal_size['Deal Size'].str.contains('B', na=False)].copy()

# Prepare empty lists to store the calculated returns, open, and close prices
ipo_returns = []
ipo_opens = []
ipo_closes = []

print(f"Found {len(filtered_df)} IPOs with 'B' in 'Deal Size'. Attempting to fetch IPO day stock data...")

for index, row in filtered_df.iterrows():
    symbol = row['Symbol']
    ipo_date_str = row['IPO Date']

    try:
        # Convert IPO Date string to datetime object
        ipo_date = pd.to_datetime(ipo_date_str)
        # Expand the window by 1 day to capture the opening cross print with 1-minute intervals
        end_date = ipo_date + pd.Timedelta(days=1)

        # Fetch data for the IPO date using yfinance with 1-minute intervals
        ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval="1m", progress=False)

        if not ticker_data.empty:
            # Get the first 'Open' and last 'Close' from the intraday data
            open_price = ticker_data['Open'].iloc[0]
            close_price = ticker_data['Close'].iloc[-1]

            # Calculate return (Close - Open) / Open
            if open_price != 0:
                return_val = (close_price - open_price) / open_price
                ipo_returns.append(return_val)
                ipo_opens.append(open_price)
                ipo_closes.append(close_price)
                print(f"    ✅ Fetched {symbol} (IPO Date: {ipo_date.strftime('%Y-%m-%d')}): Open={open_price:.2f}, Close={close_price:.2f}, Return={return_val:.2%}")
            else:
                ipo_returns.append(None)
                ipo_opens.append(None)
                ipo_closes.append(None)
                print(f"    ❌ Open price for {symbol} on {ipo_date.strftime('%Y-%m-%d')} is zero. Cannot calculate return.")
        else:
            ipo_returns.append(None)
            ipo_opens.append(None)
            ipo_closes.append(None)
            print(f"    ❌ No intraday stock data found for {symbol} on {ipo_date.strftime('%Y-%m-%d')} from Yahoo Finance.")
    except Exception as e:
        ipo_returns.append(None)
        ipo_opens.append(None)
        ipo_closes.append(None)
        print(f"    ❌ Error fetching data for {symbol} on {ipo_date_str}: {e}")

# Add the calculated returns, open, and close prices to the filtered DataFrame
filtered_df['IPO_Day_Open'] = ipo_opens
filtered_df['IPO_Day_Close'] = ipo_closes
filtered_df['IPO_Day_Return'] = ipo_returns

/tmp/ipykernel_3392/2862582422.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval="1m", progress=False)
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SARO']: YFPricesMissingError('possibly delisted; no price data found  (1m 2024-10-02 -> 2024-10-03) (Yahoo error = "1m data not available for startTime=1727875800 and endTime=1727928000. The requested range must be within the last 30 days.")')


Found 26 IPOs with 'B' in 'Deal Size'. Attempting to fetch IPO day stock data...
    ❌ No intraday stock data found for SARO on 2024-10-02 from Yahoo Finance.


/tmp/ipykernel_3392/2862582422.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval="1m", progress=False)
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['LINE']: YFPricesMissingError('possibly delisted; no price data found  (1m 2024-07-25 -> 2024-07-26) (Yahoo error = "1m data not available for startTime=1721914200 and endTime=1721966400. The requested range must be within the last 30 days.")')
/tmp/ipykernel_3392/2862582422.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval="1m", progress=False)
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['VIK']: YFPricesMissingError('possibly delisted; no price data found  (1m 2024-05-01 -> 2024-05-02) (Yahoo error = "1m data not available for s

    ❌ No intraday stock data found for LINE on 2024-07-25 from Yahoo Finance.
    ❌ No intraday stock data found for VIK on 2024-05-01 from Yahoo Finance.


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AS']: YFPricesMissingError('possibly delisted; no price data found  (1m 2024-02-01 -> 2024-02-02) (Yahoo error = "1m data not available for startTime=1706797800 and endTime=1706850000. The requested range must be within the last 30 days.")')
/tmp/ipykernel_3392/2862582422.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval="1m", progress=False)
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['MDLN']: YFPricesMissingError('possibly delisted; no price data found  (1m 2025-12-17 -> 2025-12-18) (Yahoo error = "1m data not available for startTime=1765981800 and endTime=1766034000. The requested range must be within the last 30 days.")')
/tmp/ipykernel_3392/2862582422.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, sta

...

    ❌ No intraday stock data found for FRVO on 2026-05-13 from Yahoo Finance.
    ❌ No intraday stock data found for PS on 2026-04-29 from Yahoo Finance.


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['XE']: YFPricesMissingError('possibly delisted; no price data found  (1m 2026-04-24 -> 2026-04-25) (Yahoo error = "1m data not available for startTime=1777037400 and endTime=1777089600. The requested range must be within the last 30 days.")')
/tmp/ipykernel_3392/2862582422.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval="1m", progress=False)
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ARXS']: YFPricesMissingError('possibly delisted; no price data found  (1m 2026-04-16 -> 2026-04-17) (Yahoo error = "1m data not available for startTime=1776346200 and endTime=1776398400. The requested range must be within the last 30 days.")')
/tmp/ipykernel_3392/2862582422.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, sta

    ❌ No intraday stock data found for XE on 2026-04-24 from Yahoo Finance.
    ❌ No intraday stock data found for ARXS on 2026-04-16 from Yahoo Finance.


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['MAIR']: YFPricesMissingError('possibly delisted; no price data found  (1m 2026-04-16 -> 2026-04-17) (Yahoo error = "1m data not available for startTime=1776346200 and endTime=1776398400. The requested range must be within the last 30 days.")')
/tmp/ipykernel_3392/2862582422.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval="1m", progress=False)
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FPS']: YFPricesMissingError('possibly delisted; no price data found  (1m 2026-02-05 -> 2026-02-06) (Yahoo error = "1m data not available for startTime=1770301800 and endTime=1770354000. The requested range must be within the last 30 days.")')


    ❌ No intraday stock data found for MAIR on 2026-04-16 from Yahoo Finance.
    ❌ No intraday stock data found for FPS on 2026-02-05 from Yahoo Finance.


In [8]:
print("--- Filtered DataFrame with IPO Day Returns ---")
display(filtered_df[['Symbol', 'Company Name', 'IPO Date', 'Deal Size', 'IPO_Day_Open', 'IPO_Day_Close', 'IPO_Day_Return']].head())
display(filtered_df[['Symbol', 'Company Name', 'IPO Date', 'Deal Size', 'IPO_Day_Open', 'IPO_Day_Close', 'IPO_Day_Return']].tail())

--- Filtered DataFrame with IPO Day Returns ---


,Symbol,Company Name,IPO Date,Deal Size,IPO_Day_Open,IPO_Day_Close,IPO_Day_Return
69,SARO,"StandardAero, Inc.","Oct 2, 2024",1.44B,None,None,None
115,LINE,"Lineage, Inc.","Jul 25, 2024",4.44B,None,None,None
160,VIK,Viking Holdings Ltd,"May 1, 2024",1.27B,None,None,None
209,AS,"Amer Sports, Inc.","Feb 1, 2024",1.37B,None,None,None
234,MDLN,Medline Inc.,"Dec 17, 2025",6.26B,None,None,None


,Symbol,Company Name,IPO Date,Deal Size,IPO_Day_Open,IPO_Day_Close,IPO_Day_Return
664,PS,Pershing Square Inc.,"Apr 29, 2026",1.66B,None,None,None
668,XE,"X-Energy, Inc.","Apr 24, 2026",1.02B,None,None,None
676,ARXS,"Arxis, Inc.","Apr 16, 2026",1.13B,None,None,None
677,MAIR,Madison Air Solutions Corporation,"Apr 16, 2026",2.23B,None,None,None
735,FPS,"Forgent Power Solutions, Inc.","Feb 5, 2026",1.51B,None,None,None
